---
## Setup

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
from pathlib import Path

# Locate project root and load credentials.
# parents[1] walks two levels up from notebooks/ to reach the project root,
# regardless of where VS Code starts the kernel.
PROJECT_ROOT = Path().resolve().parents[1]
load_dotenv(PROJECT_ROOT / ".env")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "news_pulse")

if not DB_PASS:
    raise ValueError("DB_PASSWORD not found in .env")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True
)

def query(sql: str) -> pd.DataFrame:
    """Run a SQL string and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print(f"Connected to: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"Project root: {PROJECT_ROOT}")

---
## Section 1 — Warehouse Row Counts

The first check after any load is confirming the row counts match what the pipeline reported. If they do not, something went wrong either in the load step or in this query.

In [ ]:
counts = query("""
    SELECT 'dim_source'           AS tbl, COUNT(*) AS row_count FROM dim_source
    UNION ALL
    SELECT 'dim_segment',                 COUNT(*) FROM dim_segment
    UNION ALL
    SELECT 'dim_entity',                  COUNT(*) FROM dim_entity
    UNION ALL
    SELECT 'dim_date',                    COUNT(*) FROM dim_date
    UNION ALL
    SELECT 'fact_articles',               COUNT(*) FROM fact_articles
    UNION ALL
    SELECT 'fact_entity_mentions',        COUNT(*) FROM fact_entity_mentions
""")

print(counts.to_string(index=False))

**Expected counts after the one-week backfill (2025-05-07 to 2025-05-13):**

| Table | Expected |
|-------|----------|
| dim_source | ~10,504 |
| dim_segment | 8 |
| dim_entity | ~923,867 |
| dim_date | 2,922 |
| fact_articles | ~832,478 |
| fact_entity_mentions | ~4,717,558 |

`dim_date` covers 2020-01-01 to 2027-12-31. It was pre-populated to cover any historical GDELT pulls and to give the live pipeline runway through 2027 without needing schema changes.

---
## Section 2 — Data Quality: fact_articles

Row counts confirm the load completed. Quality checks confirm the data inside is usable.

In [ ]:
# Null rates across every column in fact_articles.
# A high null rate on a column the analysis depends on needs to be flagged now.
null_check = query("""
    SELECT
        COUNT(*)                                         AS total_rows,
        SUM(source_id       IS NULL)                     AS null_source_id,
        SUM(date_id         IS NULL)                     AS null_date_id,
        SUM(segment_id      IS NULL)                     AS null_segment_id,
        SUM(url IS NULL OR url = '')                     AS null_url,
        SUM(seendate        IS NULL)                     AS null_seendate,
        SUM(sentiment_score IS NULL)                     AS null_sentiment_score,
        SUM(sentiment_label IS NULL)                     AS null_sentiment_label,
        SUM(language        IS NULL)                     AS null_language
    FROM fact_articles
""")

total    = null_check['total_rows'].iloc[0]
null_cols = [c for c in null_check.columns if c != 'total_rows']

null_summary = pd.DataFrame({
    'column':     null_cols,
    'null_count': [null_check[c].iloc[0] for c in null_cols],
})
null_summary['null_pct'] = (null_summary['null_count'] / total * 100).round(2)

print(f"Total rows: {total:,}")
print()
print(null_summary.to_string(index=False))

In [ ]:
# Date range and daily coverage.
# Confirms we have data across all 7 days and nothing outside the expected range.
date_range = query("""
    SELECT
        MIN(seendate)                      AS earliest_article,
        MAX(seendate)                      AS latest_article,
        COUNT(DISTINCT DATE(seendate))     AS distinct_days
    FROM fact_articles
    WHERE seendate IS NOT NULL
""")

print(date_range.to_string(index=False))

In [ ]:
# Article volume by day.
# Even distribution is expected. A day with significantly fewer articles
# may indicate skipped files during ingestion, not a loading error.
daily_volume = query("""
    SELECT
        DATE(seendate)  AS article_date,
        COUNT(*)        AS article_count
    FROM fact_articles
    WHERE seendate IS NOT NULL
    GROUP BY DATE(seendate)
    ORDER BY article_date
""")

print(daily_volume.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily_volume['article_date'].astype(str), daily_volume['article_count'],
       color='#4C72B0', edgecolor='white')
ax.set_title('Article volume by day', fontsize=13, pad=12)
ax.set_xlabel('Date')
ax.set_ylabel('Articles')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Duplicate URL check.
# INSERT IGNORE should prevent duplicates, but worth verifying directly.
dup_check = query("""
    SELECT COUNT(*) AS duplicate_urls
    FROM (
        SELECT url
        FROM fact_articles
        GROUP BY url
        HAVING COUNT(*) > 1
    ) AS dupes
""")

dup_count = dup_check['duplicate_urls'].iloc[0]
if dup_count == 0:
    print("No duplicate URLs found. Deduplication is working correctly.")
else:
    print(f"WARNING: {dup_count:,} duplicate URLs detected. Investigate before proceeding.")

---
## Section 3 — Segment Distribution

Segment routing is one of the key transforms in the pipeline. The distribution needs a close look — not just the totals, but whether the General proportion is consistent across the week and what is actually inside it.

In [ ]:
# Overall segment distribution.
seg_dist = query("""
    SELECT
        s.segment_name,
        COUNT(*)                                            AS article_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1)  AS pct
    FROM fact_articles  a
    JOIN dim_segment    s ON a.segment_id = s.segment_id
    GROUP BY s.segment_name
    ORDER BY article_count DESC
""")

print(seg_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(seg_dist['segment_name'][::-1], seg_dist['article_count'][::-1],
               color='#4C72B0', edgecolor='white')
ax.set_title('Article count by segment', fontsize=13, pad=12)
ax.set_xlabel('Articles')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, pct in zip(bars, seg_dist['pct'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height() / 2,
            f'{pct}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Segment distribution by day.
# A spike in General on one specific day could mean an unusual GDELT file,
# not a routing problem in the pipeline.
seg_by_day = query("""
    SELECT
        DATE(a.seendate)    AS article_date,
        s.segment_name,
        COUNT(*)            AS article_count
    FROM fact_articles  a
    JOIN dim_segment    s ON a.segment_id = s.segment_id
    WHERE a.seendate IS NOT NULL
    GROUP BY DATE(a.seendate), s.segment_name
    ORDER BY article_date, article_count DESC
""")

seg_pivot = seg_by_day.pivot(
    index='article_date',
    columns='segment_name',
    values='article_count'
).fillna(0).astype(int)

print(seg_pivot.to_string())

In [ ]:
# Sample of General articles.
# If source names look like real news outlets, the articles are real —
# they just did not carry GDELT theme tags that matched our segment prefixes.
# That is a routing coverage issue, not a data quality issue.
general_sample = query("""
    SELECT
        src.source_name,
        a.seendate,
        a.sentiment_label
    FROM fact_articles  a
    JOIN dim_segment    s   ON a.segment_id = s.segment_id
    JOIN dim_source     src ON a.source_id  = src.source_id
    WHERE s.segment_name = 'General'
    ORDER BY RAND()
    LIMIT 20
""")

print(general_sample.to_string(index=False))

---
## Section 4 — Entity Quality Checks

The entity pipeline cleans GDELT's raw persons and organizations fields: strips position indicators, normalises to title case, deduplicates within each article. These checks confirm the cleaning worked.

In [ ]:
# Entity type breakdown.
entity_types = query("""
    SELECT
        entity_type,
        COUNT(*)                                            AS entity_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1)  AS pct
    FROM dim_entity
    GROUP BY entity_type
    ORDER BY entity_count DESC
""")

print(entity_types.to_string(index=False))

In [ ]:
# Top 25 most mentioned entities across the full week.
# If the results are recognisable names and organisations, extraction is working.
# Garbage strings here would indicate a cleaning problem to fix upstream.
top_entities = query("""
    SELECT
        e.entity_name,
        e.entity_type,
        COUNT(*) AS mention_count
    FROM fact_entity_mentions   m
    JOIN dim_entity             e ON m.entity_id = e.entity_id
    GROUP BY e.entity_id, e.entity_name, e.entity_type
    ORDER BY mention_count DESC
    LIMIT 25
""")

print(top_entities.to_string(index=False))

In [ ]:
# Suspiciously short entity names — potential parsing noise.
# Single or two-character names usually indicate a cleaning artifact.
short_entities = query("""
    SELECT
        entity_name,
        entity_type,
        CHAR_LENGTH(entity_name) AS name_length
    FROM dim_entity
    WHERE CHAR_LENGTH(entity_name) <= 2
    ORDER BY name_length, entity_name
    LIMIT 30
""")

if short_entities.empty:
    print("No suspiciously short entity names found.")
else:
    print(f"{len(short_entities)} short entity names found:")
    print(short_entities.to_string(index=False))

In [ ]:
# Entity mention distribution.
# Most entities will appear in only one article — that is normal for news data.
# The long tail is expected. What matters for network analysis is the
# subset of entities that appear across many articles.
mention_dist = query("""
    SELECT
        mention_bucket,
        COUNT(*) AS entity_count
    FROM (
        SELECT
            entity_id,
            CASE
                WHEN COUNT(*) = 1    THEN '1 article'
                WHEN COUNT(*) <= 5   THEN '2-5 articles'
                WHEN COUNT(*) <= 20  THEN '6-20 articles'
                WHEN COUNT(*) <= 100 THEN '21-100 articles'
                ELSE '100+ articles'
            END AS mention_bucket
        FROM fact_entity_mentions
        GROUP BY entity_id
    ) AS buckets
    GROUP BY mention_bucket
    ORDER BY MIN(
        CASE mention_bucket
            WHEN '1 article'       THEN 1
            WHEN '2-5 articles'    THEN 2
            WHEN '6-20 articles'   THEN 3
            WHEN '21-100 articles' THEN 4
            ELSE 5
        END
    )
""")

print(mention_dist.to_string(index=False))

---
## Section 5 — Sentiment Distribution

Sentiment is scored on the source name column using VADER. Source names are short and often carry no sentiment words at all, so a heavy concentration at zero is expected. This section confirms VADER ran successfully on all articles and that the label split looks plausible.

In [ ]:
# Overall sentiment label distribution.
sentiment_dist = query("""
    SELECT
        sentiment_label,
        COUNT(*)                                            AS article_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1)  AS pct
    FROM fact_articles
    WHERE sentiment_label IS NOT NULL
    GROUP BY sentiment_label
    ORDER BY article_count DESC
""")

print(sentiment_dist.to_string(index=False))

colors = {'POSITIVE': '#2ca02c', 'NEUTRAL': '#aec7e8', 'NEGATIVE': '#d62728'}
bar_colors = [colors.get(label, '#4C72B0')
              for label in sentiment_dist['sentiment_label']]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(sentiment_dist['sentiment_label'], sentiment_dist['article_count'],
       color=bar_colors, edgecolor='white')
ax.set_title('Sentiment label distribution', fontsize=13, pad=12)
ax.set_ylabel('Articles')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# Sentiment score distribution.
# A spike at 0.0 is expected — short source names often return a compound
# score of exactly zero because VADER finds no sentiment words.
# This is a known limitation of scoring source names rather than article text.
sentiment_scores = query("""
    SELECT sentiment_score
    FROM fact_articles
    WHERE sentiment_score IS NOT NULL
    LIMIT 100000
""")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sentiment_scores['sentiment_score'], bins=50,
        color='#4C72B0', edgecolor='white')
ax.axvline(x=0.05,  color='green', linestyle='--', linewidth=1,
           label='POSITIVE threshold (0.05)')
ax.axvline(x=-0.05, color='red',   linestyle='--', linewidth=1,
           label='NEGATIVE threshold (-0.05)')
ax.set_title('Sentiment score distribution (sample of 100k articles)',
             fontsize=13, pad=12)
ax.set_xlabel('VADER compound score')
ax.set_ylabel('Articles')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean score:   {sentiment_scores['sentiment_score'].mean():.4f}")
print(f"Median score: {sentiment_scores['sentiment_score'].median():.4f}")
print(f"Exactly zero: {(sentiment_scores['sentiment_score'] == 0).sum():,} "
      f"({(sentiment_scores['sentiment_score'] == 0).mean() * 100:.1f}%)")

---
## Section 6 — Source Coverage

Source diversity is one of the planned analysis angles in Notebook 02. These checks confirm the source dimension loaded correctly and give a first look at outlet concentration.

In [ ]:
# Top 20 sources by article volume.
top_sources = query("""
    SELECT
        src.source_name,
        COUNT(*) AS article_count
    FROM fact_articles  a
    JOIN dim_source     src ON a.source_id = src.source_id
    GROUP BY src.source_id, src.source_name
    ORDER BY article_count DESC
    LIMIT 20
""")

print(top_sources.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_sources['source_name'][::-1], top_sources['article_count'][::-1],
        color='#4C72B0', edgecolor='white')
ax.set_title('Top 20 sources by article volume', fontsize=13, pad=12)
ax.set_xlabel('Articles')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# Source concentration.
# High concentration in a small number of outlets limits the source
# diversity analysis in Notebook 02. Worth knowing before going in.
concentration = query("""
    SELECT
        SUM(CASE WHEN rnk <= 10 THEN article_count ELSE 0 END)  AS top_10_articles,
        SUM(CASE WHEN rnk <= 50 THEN article_count ELSE 0 END)  AS top_50_articles,
        SUM(article_count)                                       AS total_articles
    FROM (
        SELECT
            source_id,
            COUNT(*) AS article_count,
            RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
        FROM fact_articles
        GROUP BY source_id
    ) AS ranked
""")

top10 = concentration['top_10_articles'].iloc[0]
top50 = concentration['top_50_articles'].iloc[0]
total = concentration['total_articles'].iloc[0]
total_sources = query("SELECT COUNT(*) AS n FROM dim_source")['n'].iloc[0]

print(f"Top 10 sources:  {top10:,} articles ({top10/total*100:.1f}% of total)")
print(f"Top 50 sources:  {top50:,} articles ({top50/total*100:.1f}% of total)")
print(f"Total sources:   {total_sources:,}")

---
## Section 7 — Verification Summary

A pass/fail verdict on the warehouse. All checks must pass before moving to Notebook 02.

In [ ]:
checks = []

article_count = query("SELECT COUNT(*) AS n FROM fact_articles")['n'].iloc[0]
checks.append((
    "Minimum article count (>500k)",
    article_count > 500_000,
    f"{article_count:,} articles"
))

seg_count = query("SELECT COUNT(*) AS n FROM dim_segment")['n'].iloc[0]
checks.append((
    "All 8 segments present",
    seg_count == 8,
    f"{seg_count} segments"
))

null_urls = query("""
    SELECT COUNT(*) AS n FROM fact_articles
    WHERE url IS NULL OR url = ''
""")['n'].iloc[0]
checks.append((
    "No null URLs",
    null_urls == 0,
    f"{null_urls} null URLs"
))

dup_urls = query("""
    SELECT COUNT(*) AS n FROM (
        SELECT url FROM fact_articles
        GROUP BY url HAVING COUNT(*) > 1
    ) AS d
""")['n'].iloc[0]
checks.append((
    "No duplicate URLs",
    dup_urls == 0,
    f"{dup_urls} duplicates"
))

mention_count = query("SELECT COUNT(*) AS n FROM fact_entity_mentions")['n'].iloc[0]
checks.append((
    "Entity mentions loaded (>1M)",
    mention_count > 1_000_000,
    f"{mention_count:,} mentions"
))

distinct_days = query("""
    SELECT COUNT(DISTINCT DATE(seendate)) AS n
    FROM fact_articles WHERE seendate IS NOT NULL
""")['n'].iloc[0]
checks.append((
    "7 days of data present",
    distinct_days >= 7,
    f"{distinct_days} distinct days"
))

print("WAREHOUSE VERIFICATION SUMMARY")
print("=" * 55)
all_passed = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}]  {name}")
    print(f"          {detail}")
print("=" * 55)
if all_passed:
    print("  Overall: ALL CHECKS PASSED")
else:
    print("  Overall: FAILURES DETECTED — review before proceeding to Notebook 02")
print("=" * 55)